# BBC Ses Arşivi — LGBM Baseline Sınıflandırma
## Tez: *Generative Artificial Intelligence Models in Audio Processing: Hybrid Approaches and Applications*

Bu notebook, 26-sınıflı BBC ses arşivi üzerinde **LightGBM baseline** modelinin tüm eğitim sürecini adım adım göstermektedir.

### Pipeline Özeti
| Adım | Açıklama |
|------|----------|
| 1 | 467-dim özellik vektörü yükleme |
| 2 | **Dosya düzeyinde** stratified train/val/test bölme (%70/%15/%15) |
| 3 | StandardScaler normalleştirme + sınıf ağırlıkları |
| 4 | LightGBM eğitimi (varsayılan parametreler) |
| 5 | 5-katlı çapraz doğrulama (dosya düzeyinde) |
| 6 | Test seti değerlendirmesi |
| 7 | Karmaşıklık matrisi |
| 8 | SHAP özellik grubu önemi |

> **Not:** Segment sızıntısını (data leakage) önlemek için bölme işlemi **dosya** düzeyinde yapılmaktadır.  
> Aynı ses dosyasının farklı segmentleri asla train ve test'e ayrılmaz.

## 1. Kütüphaneler ve Ayarlar

In [1]:
import sys, warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# Proje kök dizini (notebooks/ -> üst klasör)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

ISLENMIS  = ROOT / "data" / "processed"
CHECKPTS  = ROOT / "checkpoints"
OUTPUTS   = ROOT / "outputs"
CHECKPTS.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

SEED     = 42
N_FOLDS  = 5
np.random.seed(SEED)

print(f"Proje kök: {ROOT}")
print(f"LightGBM  : {lgb.__version__}")
print(f"Pandas    : {pd.__version__}")

Proje kök: F:\Drthesiscode
LightGBM  : 4.6.0
Pandas    : 2.3.3


## 2. Veri Yükleme

`01_extract_features.py` ile oluşturulan **467-dim özellik vektörleri** parquet formatında okunur.

Her satır = 1 ses segmenti (5 sn):
- `sinif` / `sinif_idx` — sınıf etiketi
- `dosya` — kaynak WAV dosyası yolu (bölme bu alana göre yapılır)
- `f0 … f466` — 467 boyutlu özellik vektörü

### Özellik Grupları
| Grup | Boyut | İçerik |
|------|-------|--------|
| MFCC genişletilmiş | 320 | 40 katsayı × {mean, std, skew, kurt} + Δ + ΔΔ |
| Mel filterbank enerjisi | 80 | 40 band × {mean, std} — DCT uygulanmadan |
| Spektral doku | 28 | centroid, rolloff, bw, ZCR, RMS, kontrast, düzlük, flux |
| Tonal/harmonik | 36 | Chroma STFT (12) + Tonnetz (6) × {mean, std} |
| Zamansal dinamik | 3 | onset {mean, std} + tempo (BPM) |
| **Toplam** | **467** | |

In [2]:
df = pd.read_parquet(ISLENMIS / "lgbm_ozellikler.parquet")

# Özellik sütunları: f0, f1, ... fN
ozellik_sutunlari = sorted(
    [c for c in df.columns if c.startswith("f") and c[1:].isdigit()],
    key=lambda c: int(c[1:])
)
N_DIM = len(ozellik_sutunlari)
sinif_adlari = sorted(df["sinif"].unique())
N_SINIF = len(sinif_adlari)

print(f"Toplam segment : {len(df):,}")
print(f"Sinif sayisi   : {N_SINIF}")
print(f"Ozellik boyutu : {N_DIM}-dim")
print(f"Kaynak dosya   : {df['dosya'].nunique():,} benzersiz WAV\n")

# Sınıf dağılımı
print("Sınıf bazlı segment dağılımı:")
dagılim = df.groupby("sinif").size().sort_values(ascending=False)
print(dagılim.to_string())

Toplam segment : 312,725
Sinif sayisi   : 26
Ozellik boyutu : 467-dim
Kaynak dosya   : 14,462 benzersiz WAV

Sınıf bazlı segment dağılımı:
sinif
sehir_atmosferi        45004
kara_tasitlari         27329
endustri_fabrika       27141
kalabalik_sesleri      20378
ucak_sesleri           18403
doga_atmosferi         18383
tren_sesleri           16302
gemi_tekne_sesleri     15623
spor_eglence           14917
ofis_iletisim          14569
su_sesleri             12175
cocuk_bebek_sesleri    11258
silah_patlama          10991
hava_olaylari           8836
at_sesleri              8298
saat_can_sesleri        7896
ev_ici_sesler           6443
insaat_sesleri          4362
diger_hayvanlar         4190
dini_mekan              3702
kus_sesleri             3627
acil_servis             3491
ayak_sesleri            3262
kopek_sesleri           3084
tarim_ciftlik           1819
kapi_sesleri            1242


## 3. Dosya Düzeyinde Stratified Train/Val/Test Bölme

### Neden dosya düzeyinde bölme?

Aynı ses dosyasından elde edilen segmentler (örneğin 60 saniyelik bir kayıt → 12 adet 5 saniyelik segment) birbirine çok benzer. Segment düzeyinde bölme yapılırsa, eğitim setindeki bir segmentin komşu segmenti test setine düşer ve model "ezberlemiş" gibi görünür — bu **veri sızıntısı** (data leakage) olarak adlandırılır.

**Çözüm:** Önce dosyalar sınıf oranlarını koruyacak şekilde bölünür, ardından o dosyaların tüm segmentleri ilgili bölüme atanır.

```
Dosyalar  → %70 Train  +  %15 Val  +  %15 Test
Segmentler → ilgili dosyanın bölümünü miras alır
```

In [3]:
# Dosya → sınıf eşlemesi (her dosyanın bir sınıfı var)
dosya_sinif = df.groupby("dosya")["sinif_idx"].first().reset_index()
dosyalar    = dosya_sinif["dosya"].values
siniflar    = dosya_sinif["sinif_idx"].values

# 1) %85 trainval + %15 test
d_trainval, d_test, s_trainval, _ = train_test_split(
    dosyalar, siniflar, test_size=0.15, stratify=siniflar, random_state=SEED)

# 2) %70 train + %15 val (trainval'ın %82.4'ü = toplam %70)
val_oran = 0.15 / 0.85
d_train, d_val = train_test_split(
    d_trainval, test_size=val_oran, stratify=s_trainval, random_state=SEED)

# Segment DataFrame'lerini oluştur
df_train    = df[df["dosya"].isin(d_train)]
df_val      = df[df["dosya"].isin(d_val)]
df_test     = df[df["dosya"].isin(d_test)]
df_trainval = df[df["dosya"].isin(d_trainval)]

def xy(d): return d[ozellik_sutunlari].values, d["sinif_idx"].values

X_train, y_train = xy(df_train)
X_val,   y_val   = xy(df_val)
X_test,  y_test  = xy(df_test)

print(f"{'Bolum':<10} {'Dosya':>8}  {'Segment':>9}")
print("-" * 32)
print(f"{'Train':<10} {len(d_train):>8,}  {len(df_train):>9,}")
print(f"{'Val':<10} {len(d_val):>8,}  {len(df_val):>9,}")
print(f"{'Test':<10} {len(d_test):>8,}  {len(df_test):>9,}")
print("-" * 32)
print(f"{'TOPLAM':<10} {len(dosyalar):>8,}  {len(df):>9,}")

Bolum         Dosya    Segment
--------------------------------
Train        10,122    218,258
Val           2,170     47,078
Test          2,170     47,389
--------------------------------
TOPLAM       14,462    312,725


## 4. Özellik Ölçekleme ve Sınıf Ağırlıkları

### StandardScaler
LightGBM ağaç tabanlı olduğundan ölçeklemeye teorik olarak ihtiyaç duymaz. Ancak:
- Sonraki aşamalarda kullanılacak **LSTM** ve **CNN** modelleri için scaler kaydedilir
- Bazı düzenlileştirme (regularization) parametrelerinin ölçeğe duyarlı olabileceği bilinir

Scaler **yalnızca train seti üzerinde** fit edilir; val/test'e sadece `transform` uygulanır — bu *bilgi sızıntısını* önler.

### Sınıf Ağırlıkları
Veri dengesizdir: `sehir_atmosferi` 45,000 segment, `kapi_sesleri` yalnızca 1,242 segment. `class_weight="balanced"` ile azınlık sınıfların loss'taki payı artırılır.

In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Sınıf ağırlıkları (dengesiz veri için)
sinif_idx_arr = np.unique(y_train)
agirliklar    = compute_class_weight("balanced", classes=sinif_idx_arr, y=y_train)
ag_map        = dict(zip(sinif_idx_arr, agirliklar))
sample_w_train = np.array([ag_map[y] for y in y_train])

joblib.dump(scaler, CHECKPTS / "lgbm_scaler.pkl")
print("Scaler kaydedildi:", CHECKPTS / "lgbm_scaler.pkl")
print(f"\nAgirlik araligi: {agirliklar.min():.3f} — {agirliklar.max():.3f}")
print(f"En agir sinif  : {sinif_adlari[sinif_idx_arr[np.argmax(agirliklar)]]} ({agirliklar.max():.2f})")
print(f"En hafif sinif : {sinif_adlari[sinif_idx_arr[np.argmin(agirliklar)]]} ({agirliklar.min():.2f})")

Scaler kaydedildi: F:\Drthesiscode\checkpoints\lgbm_scaler.pkl

Agirlik araligi: 0.272 — 10.776
En agir sinif  : kapi_sesleri (10.78)
En hafif sinif : sehir_atmosferi (0.27)


## 5. LightGBM Model Eğitimi

### Neden LightGBM?
- Gradient boosting tabanlı, yüksek boyutlu (467-dim) tablolu veriler için state-of-the-art
- Histogram tabanlı algoritması ile hızlı eğitim (~300K segment)
- Yerleşik özellik önemi + SHAP desteği
- Tez savunmasında CNN/AST ile karşılaştırma için **baseline** görevi görür

### Kullanılan Parametreler (Baseline — HPO öncesi)
| Parametre | Değer | Açıklama |
|-----------|-------|----------|
| `n_estimators` | 500 | Ağaç sayısı |
| `num_leaves` | 127 | Yaprak sayısı (2^7−1) |
| `max_depth` | 8 | Maksimum derinlik |
| `learning_rate` | 0.05 | Öğrenme hızı |
| `subsample` | 0.8 | Satır örnekleme oranı |
| `colsample_bytree` | 0.8 | Sütun örnekleme oranı |

In [5]:
params = dict(
    n_estimators=500, num_leaves=127, max_depth=8,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, min_child_samples=20,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

checkpoint = CHECKPTS / "lgbm_model.pkl"

if checkpoint.exists():
    # Önceden eğitilmiş model yükle (scripts/02_train_lgbm.py ile üretildi)
    model = joblib.load(checkpoint)
    print(f"Checkpoint yüklendi : {checkpoint}")
    print(f"Agac sayisi         : {model.n_estimators_}")
    print("\n[Yeniden egitmek icin checkpoint dosyasini silin]")
else:
    # İlk kez çalıştırılıyorsa eğit (uzun sürer: ~30 dk)
    print("Egitim baslıyor (scripts/02_train_lgbm.py ile yapilmasi onerilir)...")
    model = lgb.LGBMClassifier(**params)
    model.fit(
        np.vstack([X_train_s, X_val_s]),
        np.concatenate([y_train, y_val]),
        sample_weight=np.concatenate([
            sample_w_train,
            np.array([ag_map.get(y, 1.0) for y in y_val])
        ]),
    )
    joblib.dump(model, checkpoint)
    print(f"Model kaydedildi : {checkpoint}")

Checkpoint yüklendi : F:\Drthesiscode\checkpoints\lgbm_model.pkl
Agac sayisi         : 500

[Yeniden egitmek icin checkpoint dosyasini silin]


## 6. 5-Katlı Çapraz Doğrulama (Dosya Düzeyinde)

CV'nin amacı modelin **genelleme gücünü güvenilir biçimde ölçmek** ve sonuçlara güven aralığı eklemektir. Tez yazımında `macro-F1 = X.XX ± Y.YY` şeklinde raporlanır.

- Bölme işlemi her fold'da **dosya düzeyinde** yapılır
- Her fold bağımsız bir model eğitir ve değerlendirir
- Fold modellerinde `n_estimators` ve `num_leaves` küçültülür — büyük modelin bellek serialization hatasını önler

In [6]:
cv_csv = OUTPUTS / "lgbm_cv_results.csv"

if cv_csv.exists():
    # scripts/02_train_lgbm.py tarafından üretilmiş sonuçları yükle
    cv_df = pd.read_csv(cv_csv)
    print("CV sonuçları yüklendi:", cv_csv)
else:
    # Notebook içinde hızlı demo CV (2 fold, 100 ağaç)
    print("CV sonuçları bulunamadı — demo CV çalıştırılıyor (2-fold, 100 agac)...")
    dosya_sinif_tv = (df_trainval.groupby("dosya")["sinif_idx"]
                      .first().reset_index())
    dosya_arr = dosya_sinif_tv["dosya"].values
    sinif_arr = dosya_sinif_tv["sinif_idx"].values

    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
    fold_sonuclari = []
    demo_params = dict(**params, n_estimators=100, num_leaves=31)

    for fold, (tr_idx, te_idx) in enumerate(skf.split(dosya_arr, sinif_arr), 1):
        tr_d = set(dosya_arr[tr_idx]); te_d = set(dosya_arr[te_idx])
        df_tr = df_trainval[df_trainval["dosya"].isin(tr_d)]
        df_te = df_trainval[df_trainval["dosya"].isin(te_d)]
        X_tr, y_tr = xy(df_tr); X_te, y_te = xy(df_te)
        X_tr = scaler.transform(X_tr); X_te = scaler.transform(X_te)
        ags = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
        ag  = dict(zip(np.unique(y_tr), ags))
        m = lgb.LGBMClassifier(**demo_params)
        m.fit(X_tr, y_tr, sample_weight=[ag[y] for y in y_tr])
        y_pred = m.predict(X_te)
        fold_sonuclari.append({
            "fold": fold,
            "accuracy": accuracy_score(y_te, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_te, y_pred),
            "macro_f1": f1_score(y_te, y_pred, average="macro"),
            "weighted_f1": f1_score(y_te, y_pred, average="weighted"),
        })
        r = fold_sonuclari[-1]
        print(f"  Fold {fold}/2  acc={r['accuracy']:.4f}  macro-F1={r['macro_f1']:.4f}")
    cv_df = pd.DataFrame(fold_sonuclari)

print("\nCV Özeti:")
print(cv_df[["accuracy","balanced_accuracy","macro_f1","weighted_f1"]]
      .agg(["mean","std"]).round(4).to_string())

CV sonuçları yüklendi: F:\Drthesiscode\outputs\lgbm_cv_results.csv

CV Özeti:
      accuracy  balanced_accuracy  macro_f1  weighted_f1
mean    0.5832             0.5309    0.5529       0.5774
std     0.0053             0.0130    0.0155       0.0072


## 7. Test Seti Değerlendirmesi

Model artık hiç görmediği **test seti** üzerinde değerlendirilir. Bu nihai performans metriğidir ve tez raporunda bu değerler kullanılır.

- **Accuracy**: Doğru tahmin oranı
- **Balanced Accuracy**: Sınıf başına accuracy ortalaması (dengesiz veri için daha adil)
- **Macro F1**: Her sınıfın F1 puanının ağırlıksız ortalaması — tüm sınıflara eşit önem verir

In [7]:
y_pred_test = model.predict(X_test_s)

test_acc  = accuracy_score(y_test, y_pred_test)
test_bacc = balanced_accuracy_score(y_test, y_pred_test)
test_mf1  = f1_score(y_test, y_pred_test, average="macro")
test_wf1  = f1_score(y_test, y_pred_test, average="weighted")

print("=" * 55)
print(f"  Test Accuracy         : {test_acc:.4f}")
print(f"  Test Balanced Acc     : {test_bacc:.4f}")
print(f"  Test Macro F1         : {test_mf1:.4f}")
print(f"  Test Weighted F1      : {test_wf1:.4f}")
print("=" * 55)

# Per-class F1
report = classification_report(
    y_test, y_pred_test,
    target_names=sinif_adlari, output_dict=True, zero_division=0)
per_class = pd.DataFrame(report).T.loc[sinif_adlari, ["precision","recall","f1-score","support"]]
per_class = per_class.sort_values("f1-score")
print("\nPer-class F1 (dusukten yuksege):")
print(per_class.round(3).to_string())

import json
metriks = {
    "test_accuracy": test_acc, "test_balanced_accuracy": test_bacc,
    "test_macro_f1": test_mf1, "test_weighted_f1": test_wf1,
    "cv_macro_f1_mean": float(cv_df["macro_f1"].mean()),
    "cv_macro_f1_std": float(cv_df["macro_f1"].std()),
    "n_dim": N_DIM, "n_sinif": N_SINIF,
}
(OUTPUTS / "lgbm_metrics.json").write_text(json.dumps(metriks, indent=2))
print(f"\nMetrikler kaydedildi: {OUTPUTS / 'lgbm_metrics.json'}")

  Test Accuracy         : 0.6228
  Test Balanced Acc     : 0.5545
  Test Macro F1         : 0.5857
  Test Weighted F1      : 0.6176

Per-class F1 (dusukten yuksege):
                     precision  recall  f1-score  support
tarim_ciftlik            0.450   0.218     0.294    229.0
insaat_sesleri           0.598   0.255     0.357    432.0
dini_mekan               0.491   0.283     0.359    692.0
ev_ici_sesler            0.487   0.341     0.401    936.0
acil_servis              0.697   0.301     0.421    405.0
spor_eglence             0.533   0.392     0.452   2132.0
hava_olaylari            0.644   0.401     0.494   1283.0
kus_sesleri              0.637   0.416     0.503    700.0
diger_hayvanlar          0.624   0.450     0.523    653.0
doga_atmosferi           0.604   0.522     0.560   2724.0
kalabalik_sesleri        0.525   0.625     0.571   2863.0
su_sesleri               0.622   0.529     0.572   1916.0
endustri_fabrika         0.529   0.678     0.594   3836.0
tren_sesleri          

## 8. Karmaşıklık Matrisi

Karmaşıklık matrisi hangi sınıfların birbirine karıştırıldığını gösterir. Normalize edilmiş versiyonda her hücre "gerçek sınıf X iken modelin Y tahmin etme oranı"nı verir. Köşegen ne kadar parlaksa model o kadar iyi.

In [8]:
cm = confusion_matrix(y_test, y_pred_test, normalize="true")
fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=sinif_adlari, yticklabels=sinif_adlari,
            linewidths=0.3, ax=ax, annot_kws={"size": 7})
ax.set_xlabel("Tahmin edilen sinif", fontsize=12)
ax.set_ylabel("Gercek sinif", fontsize=12)
ax.set_title("Normallestirilmis Karmasiklik Matrisi — LGBM Baseline", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
cikti = OUTPUTS / "lgbm_confusion_matrix.png"
fig.savefig(cikti, dpi=150)
plt.show()
print(f"Kaydedildi: {cikti}")

Kaydedildi: F:\Drthesiscode\outputs\lgbm_confusion_matrix.png


## 9. SHAP Özellik Grubu Önemi

**SHAP (SHapley Additive exPlanations)** her özelliğin tahmine katkısını oyun teorisi temelli hesaplar. Burada 467 özellik 5 gruba toplanarak her grubun modele toplam katkısı görselleştirilir.

- Yüksek SHAP değeri → o özellik grubu tahmin için kritik
- Tez bölümünde "hangi özellikler ses sınıflandırması için en önemli?" sorusunu yanıtlar

> **Teknik not:** `shap >= 0.40` sürümünde çok sınıflı modeller için SHAP değerleri  
> `(n_örnek, n_özellik, n_sınıf)` şeklinde 3B dizi döndürür.  
> Önceki sürümlerde `n_sınıf × (n_örnek, n_özellik)` listesi dönerdi.  
> Aşağıdaki kod her iki formatı da destekler.

In [9]:
OZELLIK_GRUPLARI = {
    "MFCC genisletilmis (320)":     (0,   320),
    "Mel filterbank enerjisi (80)":  (320, 400),
    "Spektral doku (28)":           (400, 428),
    "Tonal / harmonik (36)":        (428, 464),
    "Zamansal dinamik (3)":         (464, 467),
}

# SHAP için 2000 rastgele test segmenti yeterli
idx_ornek = np.random.choice(len(X_test_s), size=min(2000, len(X_test_s)), replace=False)
X_ornek   = X_test_s[idx_ornek]

print("SHAP hesaplaniyor...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_ornek)

sv = np.array(shap_values)
# Format tespiti: yeni shap (3B) vs eski shap (liste/3B farklı eksen)
if sv.ndim == 3 and sv.shape[1] == X_ornek.shape[1]:
    # (n_ornek, n_ozellik, n_sinif)
    shap_abs = np.abs(sv).mean(axis=(0, 2))
elif sv.ndim == 3:
    # (n_sinif, n_ornek, n_ozellik)
    shap_abs = np.abs(sv).mean(axis=(0, 1))
else:
    shap_abs = np.abs(sv).mean(axis=0)

# Grup bazlı toplam
grup_onemleri = {
    ad: float(shap_abs[bas:son].sum())
    for ad, (bas, son) in OZELLIK_GRUPLARI.items()
}
toplam = sum(grup_onemleri.values())
grup_pct = {k: v / toplam * 100 for k, v in grup_onemleri.items()}

print("\nOzellik Grubu SHAP Katki Orani:")
for ad, pct in sorted(grup_pct.items(), key=lambda x: -x[1]):
    bar = "█" * int(pct / 2)
    print(f"  {ad:<38} {pct:5.1f}%  {bar}")

# Görselleştirme
fig, ax = plt.subplots(figsize=(10, 5))
renkler = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
adlar   = list(grup_pct.keys())
degerler = list(grup_pct.values())
ax.barh(adlar, degerler, color=renkler, edgecolor="white")
for i, v in enumerate(degerler):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=10)
ax.set_xlabel("Toplam SHAP katki orani (%)", fontsize=11)
ax.set_title("Ozellik Grubu Onem Analizi (SHAP) — LGBM Baseline", fontsize=12)
ax.set_xlim(0, max(degerler) * 1.18)
plt.tight_layout()
cikti = OUTPUTS / "lgbm_shap_groups.png"
fig.savefig(cikti, dpi=150)
plt.show()
print(f"Kaydedildi: {cikti}")

SHAP hesaplaniyor...



Ozellik Grubu SHAP Katki Orani:
  MFCC genisletilmis (320)                55.8%  ███████████████████████████
  Mel filterbank enerjisi (80)            17.1%  ████████
  Spektral doku (28)                      16.8%  ████████
  Tonal / harmonik (36)                    9.0%  ████
  Zamansal dinamik (3)                     1.3%  
Kaydedildi: F:\Drthesiscode\outputs\lgbm_shap_groups.png


## 10. Sonuç Özeti

Bu bölüm tez raporunda doğrudan kullanılabilecek sayıları özetler.

In [10]:
print("=" * 60)
print("  LGBM BASELINE — SONUC OZETI")
print("=" * 60)
print(f"\n  Veri")
print(f"    Toplam segment      : {len(df):,}")
print(f"    Sinif sayisi        : {N_SINIF}")
print(f"    Ozellik boyutu      : {N_DIM}-dim")
print(f"\n  5-Fold CV (dosya duzeyinde)")
print(f"    Macro F1            : {cv_df['macro_f1'].mean():.4f} +- {cv_df['macro_f1'].std():.4f}")
print(f"    Accuracy            : {cv_df['accuracy'].mean():.4f} +- {cv_df['accuracy'].std():.4f}")
print(f"\n  Test Seti")
print(f"    Macro F1            : {test_mf1:.4f}")
print(f"    Weighted F1         : {test_wf1:.4f}")
print(f"    Accuracy            : {test_acc:.4f}")
print(f"    Balanced Accuracy   : {test_bacc:.4f}")
print(f"\n  En dusuk F1 siniflari (az veri):")
en_dusuk = per_class.head(5)
for sinif, row in en_dusuk.iterrows():
    print(f"    {sinif:<35} F1={row['f1-score']:.3f}  (n={int(row['support'])})")
print("=" * 60)
print("\n  Sonraki adimlar:")
print("  1. Optuna HPO (50 deneme) ile parametre optimizasyonu")
print("  2. CNN / ResNet18 ile log-mel spectrogram bazli egitim")
print("  3. AST (Audio Spectrogram Transformer) fine-tuning")
print("  4. AudioLDM fine-tuning + classifier ile uretilen ses kalite degerlendirmesi")

  LGBM BASELINE — SONUC OZETI

  Veri
    Toplam segment      : 312,725
    Sinif sayisi        : 26
    Ozellik boyutu      : 467-dim

  5-Fold CV (dosya duzeyinde)
    Macro F1            : 0.5529 +- 0.0155
    Accuracy            : 0.5832 +- 0.0053

  Test Seti
    Macro F1            : 0.5857
    Weighted F1         : 0.6176
    Accuracy            : 0.6228
    Balanced Accuracy   : 0.5545

  En dusuk F1 siniflari (az veri):
    tarim_ciftlik                       F1=0.294  (n=229)
    insaat_sesleri                      F1=0.357  (n=432)
    dini_mekan                          F1=0.359  (n=692)
    ev_ici_sesler                       F1=0.401  (n=936)
    acil_servis                         F1=0.421  (n=405)

  Sonraki adimlar:
  1. Optuna HPO (50 deneme) ile parametre optimizasyonu
  2. CNN / ResNet18 ile log-mel spectrogram bazli egitim
  3. AST (Audio Spectrogram Transformer) fine-tuning
  4. AudioLDM fine-tuning + classifier ile uretilen ses kalite degerlendirmesi
